# 03. Normalization

### Objective
Derive rate features and apply MinMaxScaler to all telemetry features.

### I/O
- **Reads**: `../../data/processed/2_cleaned_telemetry_for_modelling.csv`
- **Writes**: `../../data/processed/3_normalized_telemetry.csv`
- **Writes**: `../../data/models/scaler_params.json`

In [1]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import json, os

INPUT_PATH  = '../../data/processed/2_cleaned_telemetry_for_modelling.csv'
OUTPUT_PATH = '../../data/processed/3_normalized_telemetry.csv'
MODEL_DIR   = '../../data/models'
os.makedirs(MODEL_DIR, exist_ok=True)

print('Libraries imported')

Libraries imported


In [2]:
df = pd.read_csv(INPUT_PATH)
print(f'Loaded {len(df)} rows')

Loaded 3240 rows


In [3]:
# v2.2 derived features - computed here before scaling so they get normalized
# along with everything else.
#
# damage_per_hit: a sniper or heavy-weapon player hits few enemies but deals
# massive damage each hit. Raw enemiesHit alone underrepresents those players,
# so I divide damageDone by hit count to get weapon-class-agnostic combat intensity.
#
# pickup_attempt_rate: true collectors walk up to items and actively attempt pickups.
# Explorers pass near items incidentally but rarely attempt them. Dividing pickupAttempts
# by time near interactables captures that deliberate collection intent.
df['damage_per_hit']       = df['damageDone']      / df['enemiesHit'].clip(lower=1)
df['pickup_attempt_rate']  = df['pickupAttempts']  / df['timeNearInteractables'].clip(lower=1)

print('Derived features added:')
print(f'  damage_per_hit range: [{df["damage_per_hit"].min():.3f}, {df["damage_per_hit"].max():.3f}]')
print(f'  pickup_attempt_rate range: [{df["pickup_attempt_rate"].min():.3f}, {df["pickup_attempt_rate"].max():.3f}]')

Derived features added:
  damage_per_hit range: [0.000, 18.667]
  pickup_attempt_rate range: [0.000, 42.000]


In [4]:
features_to_normalize = [
    'enemiesHit', 'damageDone', 'timeInCombat', 'kills',
    'itemsCollected', 'pickupAttempts', 'timeNearInteractables',
    'distanceTraveled', 'timeSprinting', 'timeOutOfCombat',
    'damage_per_hit',
    'pickup_attempt_rate',
]

available_features = [f for f in features_to_normalize if f in df.columns]
print(f'Features to normalize: {len(available_features)}')

Features to normalize: 12


Experiment B was run to understand which normalization produces the best clustering separation.
Experiment B is at: `experiments/experiment_B_clustering_config/Experiment_B_Complete.ipynb`

The grid search tested minmax_uniform, minmax_log_sparse (log1p on skewed combat features first),
and RobustScaler across 108 configurations on raw features. The actual top result from that
experiment was **minmax_log_sparse** (K=3, silhouette=0.3928) - that finding applies to
raw-feature clustering and is documented there.

This notebook normalizes features for a different purpose: computing activity score means
(score_combat, score_collect, score_explore) in notebook 04, which feed into pct percentages
that notebook 05 clusters on. For that use case, plain MinMaxScaler is sufficient - features
just need to be on the same [0,1] scale so the means are comparable across archetypes.
The log-transform advantage only applies when doing Euclidean distance directly in raw feature space.

Because of this, uniform MinMaxScaler is applied below.

In [5]:
scaler = MinMaxScaler()
df[available_features] = scaler.fit_transform(df[available_features].fillna(0))

print('Applied uniform MinMaxScaler [0, 1]')
print(df[available_features].describe())

Applied uniform MinMaxScaler [0, 1]
        enemiesHit   damageDone  timeInCombat        kills  itemsCollected  \
count  3240.000000  3240.000000   3240.000000  3240.000000     3240.000000   
mean      0.033951     0.039193      0.179007     0.029918        0.094255   
std       0.061398     0.064511      0.223110     0.058657        0.131374   
min       0.000000     0.000000      0.000000     0.000000        0.000000   
25%       0.000000     0.000000      0.000000     0.000000        0.000000   
50%       0.000000     0.000000      0.065574     0.000000        0.076923   
75%       0.053191     0.074627      0.327869     0.066667        0.153846   
max       1.000000     1.000000      1.000000     1.000000        1.000000   

       pickupAttempts  timeNearInteractables  distanceTraveled  timeSprinting  \
count     3240.000000            3240.000000       3240.000000    3240.000000   
mean         0.044323               0.084534          0.455634       0.174519   
std          0.067

In [6]:
df.to_csv(OUTPUT_PATH, index=False)
print(f'Saved to {OUTPUT_PATH}')

Saved to ../../data/processed/3_normalized_telemetry.csv


In [7]:
# Export scaler parameters so the demo frontend can normalize live player input
# using the exact same min/max values this training data was scaled with.
scaler_params = {
    'features':    list(available_features),
    'data_min':    scaler.data_min_.tolist(),
    'data_max':    scaler.data_max_.tolist(),
    'data_range':  (scaler.data_max_ - scaler.data_min_).tolist(),
    'min_value':   float(scaler.feature_range[0]),
    'max_value':   float(scaler.feature_range[1])
}

SCALER_OUT = os.path.join(MODEL_DIR, 'scaler_params.json')
with open(SCALER_OUT, 'w') as f:
    json.dump(scaler_params, f, indent=2)

print(f'Scaler parameters exported to {SCALER_OUT}')
print(f'Features included ({len(available_features)}): {available_features}')

Scaler parameters exported to ../../data/models\scaler_params.json
Features included (12): ['enemiesHit', 'damageDone', 'timeInCombat', 'kills', 'itemsCollected', 'pickupAttempts', 'timeNearInteractables', 'distanceTraveled', 'timeSprinting', 'timeOutOfCombat', 'damage_per_hit', 'pickup_attempt_rate']
